# Разбиение по датам, обучение и метрики для статьи

**Цель:** повторить сравнение методов на целых ранее не использованных датах WRF-Chem и ответить на R1-1, R2-1 и R2-3. Результат эксперимента заранее не предполагается.

Ноутбук использует только файлы из `configs/config.yaml` и существующие модули `src/`. Запускать **Restart Kernel → Run All** на сервере, из корня проекта или папки `notebooks/`. Требуется Python ≥ 3.10, зависимости из `requirements.txt`, а также `pandas`. Для первого запуска при необходимости: `%pip install -r ../requirements.txt pandas` из `notebooks/`.

По умолчанию запускаются обе нейросети, все методы из таблиц статьи и размерности **8, 16, 32, 64**. Для каждой нейросети — 5 групповых CV-прогонов и один финальный прогон: **48 обучений при одном seed**. Baseline-методы также пересчитываются на тех же группах. UMAP и TT-SVD могут работать долго. GPU используется автоматически; batch size берётся из конфига. При нехватке GPU-памяти уменьшите `BATCH_SIZE` в настройках до запуска полного эксперимента и применяйте одинаковое значение к обеим сетям.

Завершённые комбинации сохраняются и повторно не обучаются. Незавершённая комбинация после перезапуска обучается сначала. Изменение параметров, исходного кода или метаданных файлов создаёт отдельную папку результатов. Старые скрипты обучения и конфиг этот ноутбук не изменяет.

In [ ]:
from pathlib import Path
import sys, os, json, re, hashlib, random, time, gc, inspect, platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import yaml
import netCDF4 as nc
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'configs/config.yaml').is_file() and (p / 'src/models.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Откройте ноутбук внутри репозитория с configs/config.yaml и src/.')
CONFIG_PATH = PROJECT_ROOT / 'configs/config.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from models import get_model
from metrics import compute_ssim
from pca import PCACompressor
from dct import DCTCompressor
from wavelet import WaveletCompressor
from interpolation import InterpolationCompressor
from tensor_train import TTCompressor
from matrix_methods import TruncatedSVDCompressor, RandomProjectionCompressor

# Настройки эксперимента. Менять ДО просмотра test-метрик.
SPLIT_SEED = int(config['data'].get('random_seed', 42))
TRAIN_SEEDS = [SPLIT_SEED]                 # Для нескольких инициализаций: [42, 43, 44]
N_TEST_DATES, N_VAL_DATES, N_FOLDS = 2, 2, 5
LATENT_DIMS = [8, 16, 32, 64]              # Размерности из текущей статьи
AE_MODELS = ['PlainConv3DAutoencoder', 'Conv3DAutoencoder']
BASELINES = ['PCA', 'DCT', 'Wavelet', 'Interpolation',
             'TruncatedSVD', 'RandomProjection', 'UMAP', 'TT-SVD']
TT_RANKS = [(2, 4), (4, 8), (8, 8), (8, 16), (16, 16)]
RUN_CV = True
RUN_FINAL_TEST = True
EPOCHS = int(config['training']['num_epochs'])
BATCH_SIZE = int(config['training']['batch_size'])
EVAL_BATCH_SIZE = 4                       # Не влияет на обучение
PATIENCE = int(config['training']['early_stopping']['patience'])
MIN_DELTA = float(config['training']['early_stopping'].get('min_delta', 0))
LEARNING_RATE = float(config['training']['learning_rate'])
WEIGHT_DECAY = float(config['model'].get('weight_decay', 0))
DROPOUT = float(config['model'].get('dropout_rate', 0.1))
BETA = float(config['loss'].get('beta', 0.2))
EPS = 1e-7
NUM_WORKERS = 0                          # Переносимо между серверами/Jupyter
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_BASE = PROJECT_ROOT / 'results/reviewer1_grouped'
print('Config:', CONFIG_PATH, '\nDevice:', DEVICE)
print('Параметр val_split_ratio из старого конфига НЕ используется: делим целые даты.')
if min(EPOCHS, BATCH_SIZE, EVAL_BATCH_SIZE, PATIENCE) < 1:
    raise ValueError('epochs, batch sizes и patience должны быть положительными.')

## Протокол до начала обучения

1. Группа — **дата начала симуляции** из пути `Nsk_YYYY-MM-DD_...`. Все файлы и сценарии с этой датой идут вместе, включая кадры после полуночи. Файлы не пропускаются при ошибках.
2. Отсортированные уникальные даты перемешиваются `numpy.default_rng(SPLIT_SEED)`: первые 2 — **test**, следующие 2 — **validation**, остальные 7 — **train**. Это проверка на других датах из имеющегося набора, а не прогноз в будущее и не гарантия независимости метеоусловий.
3. Для CV используются только 9 development-дат (train + validation). Они разбиваются на 5 внешних folds. В каждом fold одна из оставшихся дат отдельно выделяется для early stopping; внешний fold используется только для метрик после обучения. Итоговые test-даты не участвуют ни в одном CV-fold.
4. Финальные модели обучаются на 7 train-датах, эпоха выбирается на 2 validation-датах, метрики считаются на 2 test-датах. Один и тот же план используется всеми методами. Размерности и гиперпараметры зафиксированы выше; test нельзя использовать для их последующей настройки.
5. Нормализация остаётся покадровой и по уровням, как в текущей статье. Это не обучение статистик по test. Однако для обратного перехода в физические единицы нужны min/max каждого кадра; они сохраняются отдельно. Это дополнительная информация при сжатии, а не доказанная доступность в обратной задаче.

In [ ]:
def json_write(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)

def timestamps_from_nc(ds, n):
    if 'Times' in ds.variables:
        raw = np.asarray(ds.variables['Times'][:])
        strings = []
        for row in raw:
            if np.ndim(row) == 0:
                s = row.decode() if isinstance(row, bytes) else str(row)
            elif raw.dtype.kind == 'S':
                s = b''.join(row).decode()
            else:
                s = ''.join(row.astype(str))
            strings.append(s.strip('\x00 ').replace('_', 'T'))
        values = pd.to_datetime(strings, errors='raise')
    else:
        values = None
        for name in ['XTIME', 'time', 'Time']:
            if name in ds.variables:
                v = ds.variables[name]
                units = getattr(v, 'units', '')
                if 'since' in units and v.shape == (n,):
                    decoded = nc.num2date(v[:], units, calendar=getattr(v, 'calendar', 'standard'))
                    values = pd.to_datetime([x.isoformat() for x in decoded], errors='raise')
                    break
        if values is None:
            return [None] * n, 'not_available'
    if len(values) != n or values.hasnans or (n > 1 and np.any(np.diff(values.asi8) <= 0)):
        raise ValueError('Временные метки должны соответствовать кадрам и строго возрастать.')
    return [x.isoformat() for x in values], 'netcdf'

def inventory_from_config(cfg, root):
    base = Path(cfg['data'].get('dataset_dir', ''))
    if not base.is_absolute():
        base = root / base             # Как при запуске train.py из корня проекта
    records, samples, seen_paths = [], [], set()
    expected = tuple(cfg['model']['input_shape'][1:])
    variable = cfg['data'].get('variable_name', 'co')
    for file_id, item in enumerate(cfg['data']['files']):
        path = (base / item).resolve()
        if path in seen_paths:
            raise ValueError(f'Повтор одного файла в конфиге: {path}')
        seen_paths.add(path)
        match = re.search(r'Nsk_(\d{4}-\d{2}-\d{2})_', str(item))
        if not match:
            raise ValueError(f'Не удалось определить дату запуска из пути: {item}')
        group = match.group(1)
        pd.Timestamp(group)             # Проверка календарной даты
        if not path.is_file():
            raise FileNotFoundError(f'Нет файла из конфига: {path}')
        with nc.Dataset(path) as ds:
            v = ds.variables[variable]
            if len(v.shape) != 4 or tuple(v.shape[1:]) != expected or v.shape[0] == 0:
                raise ValueError(f'{item}: CO shape={v.shape}, ожидалось (time, {expected}).')
            n = int(v.shape[0])
            ts, time_source = timestamps_from_nc(ds, n)
            known = pd.to_datetime([x for x in ts if x is not None])
            dt = np.diff(known.asi8) / 60e9 if len(known) > 1 else np.array([])
            rec = dict(file_id=file_id, file=str(item), resolved_path=str(path), group=group,
                       n_frames=n, start_index=len(samples), time_start=ts[0], time_end=ts[-1],
                       time_source=time_source, interval_min=float(dt.min()) if len(dt) else None,
                       interval_max=float(dt.max()) if len(dt) else None,
                       units=str(getattr(v, 'units', 'not_available')),
                       dimensions=list(v.dimensions), shape=list(v.shape),
                       file_size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns)
            records.append(rec)
            for local_index, stamp in enumerate(ts):
                samples.append(dict(index=len(samples), file_id=file_id, file=str(item),
                                    local_index=local_index, group=group, timestamp=stamp))
    if not records:
        raise ValueError('Список файлов пуст.')
    if len({r['units'] for r in records}) != 1:
        raise ValueError('Единицы CO различаются между файлами; сначала нужно согласовать единицы.')
    return records, pd.DataFrame(samples)

inventory, samples = inventory_from_config(config, PROJECT_ROOT)
display(pd.DataFrame(inventory)[['group', 'n_frames', 'time_start', 'time_end',
                                'interval_min', 'interval_max', 'units']])
print('Всего кадров:', len(samples), '| Дат:', samples.group.nunique())
if samples.timestamp.isna().any():
    print('ВНИМАНИЕ: часть временных меток отсутствует. Они не выдумываются из шага 10 минут.')

In [ ]:
def assert_disjoint(partitions, frame_table):
    arrays = [np.asarray(v, dtype=int) for v in partitions.values()]
    if any(len(a) == 0 or len(a) != len(np.unique(a)) for a in arrays):
        raise ValueError('Пустая выборка или повтор индексов.')
    all_ids = np.concatenate(arrays)
    if len(np.unique(all_ids)) != len(all_ids):
        raise ValueError('Пересечение кадров между выборками.')
    group_sets = [set(frame_table.iloc[a].group) for a in arrays]
    for i in range(len(arrays)):
        for j in range(i):
            if group_sets[i] & group_sets[j]:
                raise ValueError('Одна дата попала в разные выборки.')
            ti = set(frame_table.iloc[arrays[i]].timestamp.dropna())
            tj = set(frame_table.iloc[arrays[j]].timestamp.dropna())
            if ti & tj:
                raise ValueError('Совпадающие временные метки в разных группах. Нужно объединить связанные запуски.')

def build_splits(frame_table, seed, n_test, n_val, folds):
    groups = np.array(sorted(frame_table.group.unique()))
    if min(n_test, n_val) < 1 or len(groups) <= n_test + n_val:
        raise ValueError('Нужно оставить непустые train, validation и test по целым датам.')
    shuffled = np.random.default_rng(seed).permutation(groups)
    group_map = dict(test=shuffled[:n_test], validation=shuffled[n_test:n_test+n_val],
                     train=shuffled[n_test+n_val:])
    ids = lambda gs: np.flatnonzero(frame_table.group.isin(gs).to_numpy())
    holdout = {k: ids(v) for k, v in group_map.items()}
    assert_disjoint(holdout, frame_table)
    assert sorted(np.concatenate(list(holdout.values()))) == list(range(len(frame_table)))
    dev_groups = np.array(sorted(set(groups) - set(group_map['test'])))
    if not 2 <= folds <= len(dev_groups) or len(dev_groups) - int(np.ceil(len(dev_groups)/folds)) < 2:
        raise ValueError('Недостаточно development-дат для внешнего CV и внутренней validation.')
    outer_groups = np.array_split(np.random.default_rng(seed + 1).permutation(dev_groups), folds)
    cv = []
    for fold, evaluation_groups in enumerate(outer_groups, 1):
        remaining = sorted(set(dev_groups) - set(evaluation_groups))
        inner = np.random.default_rng(seed + 100 + fold).permutation(remaining)
        part = dict(train=ids(inner[1:]), validation=ids(inner[:1]), evaluation=ids(evaluation_groups))
        assert_disjoint({**part, 'test': holdout['test']}, frame_table)
        cv.append(part)
    outer_ids = np.concatenate([x['evaluation'] for x in cv])
    assert len(np.unique(outer_ids)) == len(outer_ids)
    assert set(outer_ids) == set(np.concatenate([holdout['train'], holdout['validation']]))
    return holdout, cv

holdout, cv_splits = build_splits(samples, SPLIT_SEED, N_TEST_DATES, N_VAL_DATES, N_FOLDS)
samples['split'] = ''
for label, ids in holdout.items():
    samples.loc[ids, 'split'] = label
split_table = samples.groupby(['split', 'group'], sort=True).size().rename('n_frames').reset_index()
display(split_table)
cv_table = pd.DataFrame([dict(fold=f, role=role, n_frames=len(ids),
                            n_dates=samples.iloc[ids].group.nunique(),
                            dates=', '.join(sorted(samples.iloc[ids].group.unique())))
                         for f, part in enumerate(cv_splits, 1) for role, ids in part.items()])
display(cv_table)
print('Пересечения кадров, дат и известных временных меток проверены.')

## Сохранение протокола и данных

`samples.csv` задаёт стабильный глобальный индекс: порядок файлов из конфига, затем порядок кадров в файле. `split_indices.npz` и `cv_indices.npz` можно использовать для последующих экспериментов без нового разбиения. Пропуски, masked values и NaN в полях вызывают ошибку вместо скрытого удаления кадров.

Нормализованные поля хранятся в NumPy memmap на диске (float32), поэтому все поля сразу в RAM для нейросетей не копируются. PCA/UMAP и другие обучаемые baseline-методы могут потребовать RAM для всей своей train-выборки. Старые результаты со случайным разбиением нельзя выдавать за результаты этого протокола.

In [ ]:
packages = ['torch', 'numpy', 'pandas', 'netCDF4', 'PyYAML', 'scipy', 'scikit-learn', 'matplotlib', 'PyWavelets']
if 'UMAP' in BASELINES:
    packages.append('umap-learn')
versions = {name: version(name) for name in packages}
protocol = dict(schema=1, config=config, split_seed=SPLIT_SEED, train_seeds=TRAIN_SEEDS,
                n_test_dates=N_TEST_DATES, n_val_dates=N_VAL_DATES, folds=N_FOLDS,
                latent_dims=LATENT_DIMS, ae_models=AE_MODELS, baselines=BASELINES, tt_ranks=TT_RANKS,
                run_cv=RUN_CV, run_final_test=RUN_FINAL_TEST, epochs=EPOCHS, batch_size=BATCH_SIZE,
                eval_batch_size=EVAL_BATCH_SIZE, patience=PATIENCE, min_delta=MIN_DELTA,
                learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, dropout=DROPOUT,
                loss=f'MSE + {BETA} * MAE', normalization_epsilon=EPS,
                normalization='per frame and vertical level', ssim_data_range=1.0,
                inventory=inventory, versions=versions)
protocol['source_hashes'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                             for p in sorted((PROJECT_ROOT / 'src').glob('*.py'))}
nb_path = PROJECT_ROOT / 'notebooks/reviewer1_grouped_training.ipynb'
if nb_path.exists():
    nb_source = [''.join(c['source']) for c in json.loads(nb_path.read_text())['cells']]
    protocol['notebook_source_sha256'] = hashlib.sha256(json.dumps(nb_source).encode()).hexdigest()
RUN_ID = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR = OUTPUT_BASE / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
json_write(OUTPUT_DIR / 'protocol.json', protocol)
json_write(OUTPUT_DIR / 'environment.json', dict(python=sys.version, platform=platform.platform(),
            device=str(DEVICE), cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else None, packages=versions))
samples.to_csv(OUTPUT_DIR / 'samples.csv', index=False)
pd.DataFrame(inventory).to_csv(OUTPUT_DIR / 'file_inventory.csv', index=False)
split_table.to_csv(OUTPUT_DIR / 'split_by_date.csv', index=False)
cv_table.to_csv(OUTPUT_DIR / 'cv_by_date.csv', index=False)
np.savez(OUTPUT_DIR / 'split_indices.npz', **holdout)
np.savez(OUTPUT_DIR / 'cv_indices.npz', **{f'fold_{f}_{role}': ids
         for f, part in enumerate(cv_splits, 1) for role, ids in part.items()})
print('Результаты:', OUTPUT_DIR)

def prepare_cache():
    shape = (len(samples), *config['model']['input_shape'][1:])
    done = OUTPUT_DIR / 'cache_complete.json'
    if not done.exists():
        values = np.lib.format.open_memmap(OUTPUT_DIR / 'normalized.npy', mode='w+', dtype='float32', shape=shape)
        minima = np.lib.format.open_memmap(OUTPUT_DIR / 'minima.npy', mode='w+', dtype='float64', shape=shape[:2])
        spans = np.lib.format.open_memmap(OUTPUT_DIR / 'spans.npy', mode='w+', dtype='float64', shape=shape[:2])
        for rec in inventory:
            with nc.Dataset(rec['resolved_path']) as ds:
                var = ds.variables[config['data'].get('variable_name', 'co')]
                for j in range(rec['n_frames']):
                    field = var[j]
                    if np.ma.is_masked(field) or not np.isfinite(field).all():
                        raise ValueError(f'Masked/NaN/Inf: {rec["file"]}, frame={j}')
                    field = np.asarray(field, dtype=np.float64)
                    lo, hi = field.min(axis=(1, 2)), field.max(axis=(1, 2))
                    span = hi - lo
                    i = rec['start_index'] + j
                    values[i] = (field - lo[:, None, None]) / (span[:, None, None] + EPS)
                    minima[i], spans[i] = lo, span
            print('Нормализован:', rec['group'], rec['n_frames'])
        values.flush(); minima.flush(); spans.flush()
        json_write(done, {'shape': list(shape), 'run_id': RUN_ID})
    return tuple(np.load(OUTPUT_DIR / name, mmap_mode='r') for name in ['normalized.npy', 'minima.npy', 'spans.npy'])

data, minima, spans = prepare_cache()
print('Data:', data.shape, '| disk GiB:', round(data.nbytes / 2**30, 2))

## Одинаковое обучение обеих сетей

Используется **MSE + β·MAE без весов по высоте**, как записано в формуле текущей статьи. `loss.height_weight` старого конфига здесь не применяется. Значение β и остальные параметры фиксированы конфигом и ячейкой настроек; новый Optuna-подбор этот эксперимент не выполняет.

Epoch loss усредняется по числу кадров, а не по числу batch. Лучшие веса выбираются только по validation loss. Внешние CV-данные и финальный test не передаются в функцию обучения. Один seed не оценивает разброс по инициализациям; для этого увеличьте `TRAIN_SEEDS` до запуска.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

class FieldDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=int)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, j):
        return torch.from_numpy(np.array(data[self.indices[j]], dtype=np.float32)).unsqueeze(0)

def make_loader(indices, batch_size, shuffle=False, seed=0):
    return DataLoader(FieldDataset(indices), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
                      generator=torch.Generator().manual_seed(seed))

def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def epoch_pass(model, loader, optimizer=None):
    model.train(optimizer is not None)
    totals = np.zeros(3, dtype=float)
    count = 0
    with torch.set_grad_enabled(optimizer is not None):
        for target in loader:
            target = target.to(DEVICE)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
            rec, _ = model(target)
            mse = (rec - target).square().mean()
            mae = (rec - target).abs().mean()
            loss = mse + BETA * mae
            if not torch.isfinite(loss):
                raise FloatingPointError('Нечисловой loss. Результат этого запуска не сохраняется как завершённый.')
            if optimizer is not None:
                loss.backward(); optimizer.step()
            totals += np.array([loss.item(), mse.item(), mae.item()]) * len(target)
            count += len(target)
    return totals / count

def train_ae(name, dim, train_ids, validation_ids, seed, run_dir):
    seed_everything(seed)
    model = get_model(name, latent_dim=dim, input_shape=(1, *data.shape[1:]),
                      dropout_rate=DROPOUT, use_attention=True).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler_cfg = config['training'].get('scheduler', {})
    scheduler = (torch.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_cfg['step_size'], gamma=scheduler_cfg['gamma'])
                 if scheduler_cfg.get('use_scheduler', False) else None)
    train_loader = make_loader(train_ids, BATCH_SIZE, True, seed)
    val_loader = make_loader(validation_ids, BATCH_SIZE)
    best, best_epoch, bad_epochs, history = float('inf'), 0, 0, []
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    sync(); started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        tr = epoch_pass(model, train_loader, optimizer)
        va = epoch_pass(model, val_loader)
        history.append(dict(epoch=epoch, train_loss=tr[0], train_mse=tr[1], train_mae=tr[2],
                            val_loss=va[0], val_mse=va[1], val_mae=va[2], lr=optimizer.param_groups[0]['lr']))
        if best - va[0] > MIN_DELTA:
            best, best_epoch, bad_epochs = float(va[0]), epoch, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(state, run_dir / 'best_weights.pt')
        else:
            bad_epochs += 1
        if scheduler is not None:
            scheduler.step()
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        if epoch == 1 or epoch % 5 == 0 or bad_epochs >= PATIENCE:
            print(f'{name} d={dim} epoch={epoch}: train={tr[0]:.6g}, val={va[0]:.6g}', flush=True)
        if bad_epochs >= PATIENCE:
            break
    sync()
    seconds = time.perf_counter() - started
    peak = int(torch.cuda.max_memory_allocated()) if DEVICE.type == 'cuda' else None
    model.load_state_dict(torch.load(run_dir / 'best_weights.pt', map_location=DEVICE, weights_only=True))
    model.eval()
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    hist = pd.DataFrame(history)
    for ax, metric in zip(axes, ['loss', 'mse', 'mae']):
        ax.plot(hist.epoch, hist['train_' + metric], label='train')
        ax.plot(hist.epoch, hist['val_' + metric], label='validation')
        ax.set(xlabel='Epoch', ylabel=metric); ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(run_dir / 'learning_curves.png', dpi=180); plt.close(fig)
    info = dict(best_epoch=best_epoch, best_val_loss=best, epochs_run=len(history), training_sec=seconds,
                parameter_count=sum(p.numel() for p in model.parameters()),
                weights_bytes=(run_dir / 'best_weights.pt').stat().st_size, training_peak_gpu_bytes=peak)
    return model, info

## Метрики без усреднения ошибок отдельных batch

Основные таблицы: **MSE, MAE, RMSE, относительная L2-ошибка и SSIM** на нормализованных полях. SSIM вычисляется существующей функцией проекта по двумерным срезам с фиксированным `data_range=1` для всех методов. Относительная L2 для набора — корень из отношения суммарной квадратичной ошибки к суммарной энергии исходных полей, а не среднее отношений отдельных кадров.

Дополнительно сохраняются ошибки по датам и уровням, а также MSE/MAE/RMSE/relative L2 после обратного масштабирования. Для постоянного уровня при нормализации сохраняется нуль, а при обратном преобразовании — исходная константа. Физические поля восстанавливаются из float32 нормализованных полей и сохранённых min/max; возможна малая погрешность округления. Нормализованные и физические метрики не смешиваются.

In [ ]:
def inverse_scale(arr, ids):
    lo = np.asarray(minima[ids])[:, :, None, None]
    span = np.asarray(spans[ids])[:, :, None, None]
    return np.where(span > 0, np.asarray(arr, dtype=np.float64) * (span + EPS) + lo, lo)

def aggregate_frames(frame):
    out = {'n_frames': int(len(frame))}
    for prefix in ['', 'physical_']:
        mse = float(frame[prefix + 'mse'].mean())
        energy = float(frame[prefix + 'signal_mean_square'].sum())
        sse = float(frame[prefix + 'mse'].sum())
        out[prefix + 'mse'] = mse
        out[prefix + 'mae'] = float(frame[prefix + 'mae'].mean())
        out[prefix + 'rmse'] = float(np.sqrt(mse))
        out[prefix + 'relative_l2'] = float(np.sqrt(sse / energy)) if energy > 0 else None
    out['ssim'] = float(frame.ssim.mean())
    return out

def evaluate_predictor(predict, evaluation_ids, run_dir):
    rows, height_sse, height_sae, height_psse = [], np.zeros(data.shape[1]), np.zeros(data.shape[1]), np.zeros(data.shape[1])
    started = time.perf_counter()
    for begin in range(0, len(evaluation_ids), EVAL_BATCH_SIZE):
        ids = evaluation_ids[begin:begin + EVAL_BATCH_SIZE]
        target = np.array(data[ids], dtype=np.float64)
        rec = np.asarray(predict(target.astype(np.float32)), dtype=np.float64)
        if rec.shape != target.shape or not np.isfinite(rec).all():
            raise ValueError('Некорректная форма или NaN/Inf в реконструкции.')
        phys_target, phys_rec = inverse_scale(target, ids), inverse_scale(rec, ids)
        err, perr = rec - target, phys_rec - phys_target
        height_sse += np.sum(err**2, axis=(0, 2, 3))
        height_sae += np.sum(np.abs(err), axis=(0, 2, 3))
        height_psse += np.sum(perr**2, axis=(0, 2, 3))
        for k, idx in enumerate(ids):
            rows.append(dict(index=int(idx), group=samples.iloc[idx].group,
                             mse=float(np.mean(err[k]**2)), mae=float(np.mean(np.abs(err[k]))),
                             signal_mean_square=float(np.mean(target[k]**2)),
                             ssim=float(compute_ssim(target[k], rec[k], data_range=1.0)),
                             physical_mse=float(np.mean(perr[k]**2)), physical_mae=float(np.mean(np.abs(perr[k]))),
                             physical_signal_mean_square=float(np.mean(phys_target[k]**2))))
        if begin == 0:
            # Первый кадр по индексу, не выбранный по лучшему качеству.
            np.savez_compressed(run_dir / 'example.npz', index=int(ids[0]), target=target[0], reconstructed=rec[0])
            levels = sorted(set([0, data.shape[1] // 2, data.shape[1] - 1]))
            fig, axes = plt.subplots(len(levels), 3, figsize=(10, 3 * len(levels)), squeeze=False)
            for axrow, level in zip(axes, levels):
                panels = [target[0, level], rec[0, level], np.abs(err[0, level])]
                for j, (ax, panel) in enumerate(zip(axrow, panels)):
                    im = ax.imshow(panel, origin='lower', **({'vmin': 0, 'vmax': 1} if j < 2 else {'vmin': 0}))
                    ax.set_title(['Target', 'Reconstruction', 'Absolute error'][j] + f', level {level}')
                    fig.colorbar(im, ax=ax, shrink=.8)
            fig.tight_layout(); fig.savefig(run_dir / 'reconstruction.png', dpi=160); plt.close(fig)
    frame = pd.DataFrame(rows)
    frame.to_csv(run_dir / 'per_frame.csv', index=False)
    per_date = pd.DataFrame([dict(group=g, **aggregate_frames(part)) for g, part in frame.groupby('group')])
    per_date.to_csv(run_dir / 'per_date.csv', index=False)
    denominator = len(evaluation_ids) * data.shape[2] * data.shape[3]
    pd.DataFrame(dict(level=np.arange(data.shape[1]), mse=height_sse/denominator,
                      mae=height_sae/denominator, physical_mse=height_psse/denominator)).to_csv(run_dir / 'per_level.csv', index=False)
    result = aggregate_frames(frame)
    result['evaluation_total_sec'] = time.perf_counter() - started  # Включает метрики и запись, не latency модели
    return result

def benchmark_ae(model, validation_ids):
    # Только validation; batch=1, данные уже на устройстве, без I/O и расчёта метрик.
    x = torch.from_numpy(np.array(data[validation_ids[:1]], dtype=np.float32)).unsqueeze(1).to(DEVICE)
    def encode():
        return model.fc_encoder(model.encoder_conv(x).flatten(1))
    with torch.inference_mode():
        z = encode()
        def decode():
            field = model.fc_decoder(z).view(1, *model.encoder_output_shape)
            field = model.decoder_conv(field)
            return torch.nn.functional.interpolate(field, size=data.shape[1:], mode='trilinear', align_corners=False)
        for _ in range(5):
            encode(); decode()
        timing = {}
        for label, fn in [('encode_ms', encode), ('decode_ms', decode)]:
            sync(); start = time.perf_counter()
            for _ in range(20):
                fn()
            sync(); timing[label] = (time.perf_counter() - start) * 1000 / 20
    timing['benchmark_batch_size'] = 1
    return timing

## Baseline-методы и запуск экспериментов

PCA, SVD, random projection и UMAP обучаются только на соответствующем train-наборе. UMAP использует явные `random_state`, `transform_seed`, `n_jobs=1`, `init='random'` и KNN-декодер, обученный только на train; это задано здесь явно, поскольку текущий класс проекта игнорирует переданный seed. DCT, wavelet, интерполяция и TT-SVD сжимают каждый кадр отдельно; их `fit` задаёт только форму.

Размерность/ранги здесь — параметры метода, **не доказательство равного объёма хранения**. Этот ноутбук закрывает эксперимент с разбиением и пересчитывает качество; он не подменяет отдельное сравнение по реальным байтам из R1-6 и R2-5.

In [ ]:
class SeededUMAP:
    def __init__(self, dim, seed):
        from umap import UMAP
        from sklearn.neighbors import KNeighborsRegressor
        self.dim, self.seed, self.UMAP, self.KNN = dim, seed, UMAP, KNeighborsRegressor
    def fit(self, array, verbose=False):
        self.shape = array.shape[1:]
        matrix = array.reshape(len(array), -1)
        if len(array) < 4:
            raise ValueError('UMAP требует минимум 4 train-кадра в этом протоколе.')
        self.model = self.UMAP(n_components=self.dim, n_neighbors=min(15, len(array)-1),
                              min_dist=.1, metric='euclidean', random_state=self.seed,
                              transform_seed=self.seed, n_jobs=1, init='random')
        embedding = self.model.fit_transform(matrix)
        self.decoder = self.KNN(n_neighbors=min(5, len(array)), weights='distance', n_jobs=1).fit(embedding, matrix)
        return self
    def transform(self, array):
        return self.model.transform(array.reshape(len(array), -1))
    def reconstruct(self, z):
        return self.decoder.predict(z).reshape(len(z), *self.shape)

def baseline_model(name, setting, seed):
    if name == 'TT-SVD':
        return TTCompressor(tt_ranks=setting)
    if name == 'UMAP':
        return SeededUMAP(setting, seed)
    classes = dict(PCA=PCACompressor, DCT=DCTCompressor, Wavelet=WaveletCompressor,
                   Interpolation=InterpolationCompressor, TruncatedSVD=TruncatedSVDCompressor,
                   RandomProjection=RandomProjectionCompressor)
    kwargs = {'n_components': setting}
    if name in ['TruncatedSVD', 'RandomProjection']:
        kwargs['random_state'] = seed
    return classes[name](**kwargs)

def silent_call(method, *args):
    return method(*args, **({'verbose': False} if 'verbose' in inspect.signature(method).parameters else {}))

def run_experiment(stage, fold, part, model_name, setting, seed):
    tag = str(setting).replace(' ', '').replace('(', '').replace(')', '').replace(',', 'x')
    run_dir = OUTPUT_DIR / 'runs' / f'{stage}_fold{fold}_{model_name}_d{tag}_seed{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    done = run_dir / 'result.json'
    if done.exists():
        print('Уже завершён:', run_dir.name, flush=True)
        return json.loads(done.read_text())
    assert_disjoint(part, samples)
    np.savez(run_dir / 'indices.npz', **part)
    print('\nЗапуск:', run_dir.name, '| train/val/eval:', [len(part[k]) for k in ['train', 'validation', 'evaluation']], flush=True)
    seed_everything(seed)
    info = {}
    if model_name in AE_MODELS:
        model, info = train_ae(model_name, setting, part['train'], part['validation'], seed, run_dir)
        info.update(benchmark_ae(model, part['validation']))
        def predict(batch):
            with torch.inference_mode():
                rec, _ = model(torch.from_numpy(batch).unsqueeze(1).to(DEVICE))
                return rec.cpu().numpy()[:, 0]
    else:
        model = baseline_model(model_name, setting, seed)
        if model_name in ['PCA', 'TruncatedSVD'] and setting > min(len(part['train']), int(np.prod(data.shape[1:]))):
            raise ValueError(f'{model_name}: setting={setting} превышает доступный rank; поправьте план до полного запуска.')
        started = time.perf_counter()
        fit_ids = part['train'] if model_name in ['PCA', 'TruncatedSVD', 'RandomProjection', 'UMAP'] else part['train'][:1]
        fit_data = np.array(data[fit_ids], dtype=np.float32)
        silent_call(model.fit, fit_data)
        del fit_data
        info['fit_sec'] = time.perf_counter() - started
        def predict(batch):
            return silent_call(model.reconstruct, silent_call(model.transform, batch))
    metrics = evaluate_predictor(predict, part['evaluation'], run_dir)
    result = dict(stage=stage, fold=fold, model=model_name, setting=tag, seed=seed,
                  n_train=len(part['train']), n_validation=len(part['validation']),
                  evaluation_dates=', '.join(sorted(samples.iloc[part['evaluation']].group.unique())),
                  physical_units=inventory[0]['units'], **info, **metrics)
    # Маркер завершения появляется только после всех метрик и файлов.
    json_write(done, result)
    del model, predict
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

def method_plan():
    return ([(name, dim) for name in AE_MODELS for dim in LATENT_DIMS]
            + [(name, setting) for name in BASELINES for setting in (TT_RANKS if name == 'TT-SVD' else LATENT_DIMS)])

planned = []
if RUN_CV:
    planned.extend(('cv', fold, part) for fold, part in enumerate(cv_splits, 1))
if RUN_FINAL_TEST:
    planned.append(('test', 0, dict(train=holdout['train'], validation=holdout['validation'], evaluation=holdout['test'])))
if not planned:
    raise ValueError('Не выбран ни CV, ни финальный test.')
print('Всего комбинаций:', len(planned) * len(method_plan()) * len(TRAIN_SEEDS))

In [ ]:
# Основная долгая ячейка. CV выполняется до любых test-метрик.
results = []
for stage, fold, part in planned:
    for seed in TRAIN_SEEDS:
        for name, setting in method_plan():
            results.append(run_experiment(stage, fold, part, name, setting, seed))
            pd.DataFrame(results).to_csv(OUTPUT_DIR / 'metrics_all_runs.csv', index=False)
metrics = pd.DataFrame(results)
print('Завершено комбинаций:', len(metrics))
display(metrics[['stage', 'fold', 'model', 'setting', 'seed', 'mse', 'relative_l2', 'ssim']])

## Таблицы для статьи

`cv_summary.csv` содержит mean и **sample standard deviation (`ddof=1`) по внешним folds**, отдельно для каждого seed. Это описательный разброс по датам, не доверительный интервал: обучающие части folds пересекаются. `test_metrics.csv` — ошибки по всем test-кадрам для каждой заранее заданной модели/размерности/seed. `metrics_by_date.csv` позволяет увидеть неодинаковое качество на отдельных датах. Кадры одной даты не трактуются как независимые повторения.

Таблицы по размерностям показывают качество представления и не решают вопрос одинакового бюджета памяти. Test-даты нельзя менять после просмотра этих таблиц. Если качество снизится относительно старого случайного split, это нужно отразить в статье.

In [ ]:
metric_cols = ['mse', 'mae', 'rmse', 'relative_l2', 'ssim',
               'physical_mse', 'physical_mae', 'physical_rmse', 'physical_relative_l2']
cv_metrics = metrics[metrics.stage == 'cv']
test_metrics = metrics[metrics.stage == 'test']
if len(cv_metrics):
    cv_summary = cv_metrics.groupby(['model', 'setting', 'seed'])[metric_cols].agg(['mean', 'std', 'count'])
    cv_summary.columns = ['_'.join(col) for col in cv_summary.columns]
    cv_summary = cv_summary.reset_index()
    cv_summary.to_csv(OUTPUT_DIR / 'cv_summary.csv', index=False)
    display(cv_summary[['model', 'setting', 'seed', 'mse_mean', 'mse_std', 'relative_l2_mean', 'ssim_mean']])
test_metrics.to_csv(OUTPUT_DIR / 'test_metrics.csv', index=False)
date_frames = []
for file in sorted((OUTPUT_DIR / 'runs').glob('*/result.json')):
    row = json.loads(file.read_text())
    df = pd.read_csv(file.parent / 'per_date.csv')
    for key in ['stage', 'fold', 'model', 'setting', 'seed']:
        df[key] = row[key]
    date_frames.append(df)
by_date = pd.concat(date_frames, ignore_index=True)
by_date.to_csv(OUTPUT_DIR / 'metrics_by_date.csv', index=False)
for seed in TRAIN_SEEDS:
    final = test_metrics[test_metrics.seed == seed]
    if len(final):
        display(final[['model', 'setting', 'mse', 'relative_l2', 'ssim']])
        for metric in ['mse', 'relative_l2', 'ssim']:
            table = final[final.model != 'TT-SVD'].pivot(index='model', columns='setting', values=metric)
            table = table.reindex(columns=[str(d) for d in LATENT_DIMS])
            table.to_csv(OUTPUT_DIR / f'test_{metric}_seed{seed}.csv')
            (OUTPUT_DIR / f'test_{metric}_seed{seed}.tex').write_text(table.to_latex(float_format=lambda x: f'{x:.6g}'))
        tt = final[final.model == 'TT-SVD']
        if len(tt):
            (OUTPUT_DIR / f'test_tt_seed{seed}.tex').write_text(tt[['setting', 'mse', 'relative_l2', 'ssim']].to_latex(index=False, float_format=lambda x: f'{x:.6g}'))

# Парное сравнение нейросетей на одинаковых folds/seed/размерностях, без отбора удачных запусков.
plain = metrics[metrics.model == 'PlainConv3DAutoencoder']
sam = metrics[metrics.model == 'Conv3DAutoencoder']
paired = plain.merge(sam, on=['stage', 'fold', 'setting', 'seed'], suffixes=('_plain', '_sam'))
for metric in ['mse', 'relative_l2', 'ssim']:
    paired['delta_' + metric + '_sam_minus_plain'] = paired[metric + '_sam'] - paired[metric + '_plain']
paired.to_csv(OUTPUT_DIR / 'paired_ae_comparison.csv', index=False)
display(paired[['stage', 'fold', 'setting', 'seed', 'delta_mse_sam_minus_plain', 'delta_ssim_sam_minus_plain']])

In [ ]:
if len(cv_metrics):
    for seed in TRAIN_SEEDS:
        subset = cv_metrics[cv_metrics.seed == seed]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, metric in zip(axes, ['mse', 'relative_l2', 'ssim']):
            for model_name in AE_MODELS:
                sub = subset[subset.model == model_name].copy()
                sub['dimension'] = sub.setting.astype(int)
                stat = sub.groupby('dimension')[metric].agg(['mean', 'std'])
                ax.errorbar(stat.index, stat['mean'], yerr=stat['std'], marker='o', capsize=4, label=model_name)
            ax.set(xlabel='Latent dimension', ylabel=metric, title='Grouped CV: mean ± fold SD')
            ax.grid(alpha=.2)
        axes[0].legend(fontsize=8)
        fig.tight_layout(); fig.savefig(OUTPUT_DIR / f'cv_metrics_seed{seed}.png', dpi=220); plt.show()
if len(test_metrics):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric in zip(axes, ['mse', 'relative_l2', 'ssim']):
        for (name, seed), group in test_metrics[test_metrics.model != 'TT-SVD'].groupby(['model', 'seed']):
            group = group.assign(dimension=group.setting.astype(int)).sort_values('dimension')
            ax.plot(group.dimension, group[metric], marker='o', label=f'{name}, seed={seed}')
        ax.set(xlabel='Latent dimension (not storage cost)', ylabel=metric, title='Held-out dates')
        ax.grid(alpha=.2)
    axes[-1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'test_metrics.png', dpi=220); plt.show()

In [ ]:
def md_table(frame):
    cols = list(frame.columns)
    lines = ['| ' + ' | '.join(map(str, cols)) + ' |', '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for row in frame.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join('' if pd.isna(v) else str(v) for v in row) + ' |')
    return '\n'.join(lines)

counts = {k: len(v) for k, v in holdout.items()}
dates = {k: ', '.join(sorted(samples.iloc[v].group.unique())) for k, v in holdout.items()}
known_intervals = {float(rec[key]) for rec in inventory for key in ['interval_min', 'interval_max'] if rec[key] is not None}
timing_note = ('All recorded consecutive output intervals were 10 minutes.'
               if known_intervals == {10.0} and not samples.timestamp.isna().any()
               else 'Output timestamps and intervals are reported in file_inventory.csv; temporal metadata require checking where absent.')
report = f'''# Grouped-date experiment

Run ID: {RUN_ID}

## Dataset and split (draft Methods text)

The dataset contained {len(samples)} three-dimensional CO fields from {samples.group.nunique()} simulation start dates. Each field had shape {tuple(data.shape[1:])}. {timing_note}

All frames associated with the same simulation start date were assigned to one group, including frames extending beyond midnight and any files sharing that start date. Sorted unique dates were permuted using NumPy default_rng with seed {SPLIT_SEED}. The first {N_TEST_DATES} dates were reserved for testing, the next {N_VAL_DATES} for validation, and the remaining dates for training. The final split comprised {counts['train']} training, {counts['validation']} validation and {counts['test']} test frames. Dates and known timestamps did not overlap across partitions.

Training dates: {dates['train']}.

Validation dates: {dates['validation']}.

Test dates: {dates['test']}.

This design evaluates generalization to held-out dates within the available simulation collection, rather than chronological forecasting or independence of meteorological conditions.

## Per-date counts

{md_table(split_table)}

## Cross-validation protocol

External test dates were excluded from cross-validation. The development dates were divided into {N_FOLDS} outer folds at the group level. For each outer fold, one remaining development date was held out for early stopping, and all other remaining dates were used for fitting. Outer-fold reconstruction metrics were evaluated only after restoring the weights selected on the inner validation date. The exact train, inner-validation and outer-evaluation dates and frame counts are recorded in cv_by_date.csv. CV summaries report the unweighted mean and sample standard deviation across outer folds for each training seed; folds have overlapping training data, so this standard deviation is not a confidence interval.

## Training and metrics

The autoencoders were trained with MSE + {BETA} MAE, Adam (learning rate {LEARNING_RATE}, weight decay {WEIGHT_DECAY}), dropout {DROPOUT}, batch size {BATCH_SIZE}, and at most {EPOCHS} epochs. Early stopping used validation loss with patience {PATIENCE} and min_delta {MIN_DELTA}. Seeds: {TRAIN_SEEDS}. The protocol, including scheduler settings, is stored in protocol.json. Fixed hyperparameters were used; this notebook did not perform a new hyperparameter search.

Min–max normalization was applied separately to each field and vertical level with epsilon {EPS}. MSE, MAE, RMSE, relative L2 and slice-wise SSIM (data_range=1) were evaluated on normalized fields. MSE, MAE, RMSE and relative L2 were also computed after inverse scaling using stored per-frame, per-level minima and ranges. Reported CO units from NetCDF: {inventory[0]['units']}. Missing units must be resolved before publication. The scaling parameters are required side information.

## Results and scope

Completed runs: {len(metrics)}. CV enabled: {RUN_CV}. Final test enabled: {RUN_FINAL_TEST}.

All measured outcomes are retained in metrics_all_runs.csv, cv_summary.csv (when CV is enabled), test_metrics.csv and metrics_by_date.csv. Test tables are also exported as CSV and LaTeX. paired_ae_comparison.csv contains SAM3D minus plain-CAE differences, including negative or unfavourable results. The best test configuration is not automatically selected.

These runs replace the old random-frame results only after author review. They do not establish inverse-problem stability or fair compression at equal byte budgets. The sensitivity/Jacobian analysis and full storage-cost comparison remain separate tasks. With only {N_TEST_DATES} test dates, conclusions about unseen atmospheric conditions must remain limited.
'''
(OUTPUT_DIR / 'article_report.md').write_text(report)
display(Markdown(report))
print('\nДля разбора результатов пришлите article_report.md, metrics_all_runs.csv, cv_summary.csv,')
print('test_metrics.csv, metrics_by_date.csv, paired_ae_comparison.csv и protocol.json из', OUTPUT_DIR)

## Что проверить перед переносом в статью

- Число кадров и временные интервалы соответствуют реальным файлам; сведения о неизвестных timestamps/единицах дополнены.
- Все запланированные комбинации завершены. Ошибка метода останавливает выполнение, а не скрывается как успешный результат.
- Таблицы статьи обновлены согласованно: MSE, relative L2, SSIM, TT-SVD, CV, иллюстрации и текст выводов.
- Улучшение SAM3D оценивается по парным результатам на одинаковых датах, а не по самому удачному fold.
- Новый split проверяет другие даты; наличие сезонного/метеорологического сходства и ограниченный размер набора остаются ограничениями.
- R1-1/R2-1 закрываются после фактического запуска и внесения чисел в статью, не только после подготовки ноутбука.